In [ ]:
# -----------------------------------------------------------
# Kaggle / Colab Environment Setup & Core Functions
# -----------------------------------------------------------
# This notebook is standalone and can be uploaded directly to 
# Kaggle or Google Colab without needing any external source code!
# Just upload X_scaled.csv and X_unscaled.csv to your environment.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from IPython.display import display, HTML
import os

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

# Configuration Constants
class config:
    ID_COLS = [
        "event_id", "match_id", "season_id",
        "team_id", "team_name", "player_id", "player_name",
        "period", "minute", "second", "play_pattern",
    ]
    RANDOM_STATE = 42
    K_RANGE = range(2, 11)
    KMEANS_N_INIT = 10
    DISTANCE_METRIC = "minkowski"

# -----------------------------------------------------------
# Clustering Functions
# -----------------------------------------------------------
def run_kmeans(X, k):
    model = KMeans(
        n_clusters=k,
        random_state=config.RANDOM_STATE,
        n_init=config.KMEANS_N_INIT,
    )
    labels = model.fit_predict(X)
    inertia = model.inertia_
    return labels, inertia, model

def run_kmeans_sweep(X, k_range=config.K_RANGE):
    results = {}
    for k in k_range:
        results[k] = run_kmeans(X, k)
    return results

def describe_centroids(model, X_unscaled, labels):
    df = X_unscaled.copy()
    df["Cluster"] = labels
    return df.groupby("Cluster").mean()

# -----------------------------------------------------------
# Selection Metrics Functions
# -----------------------------------------------------------
def elbow_curve(X, k_range=config.K_RANGE):
    results = run_kmeans_sweep(X, k_range)
    data = [{"k": k, "inertia": res[1]} for k, res in results.items()]
    return pd.DataFrame(data)

def suggest_k_elbow(curve):
    n_points = len(curve)
    if n_points < 3:
        return curve["k"].iloc[0]
        
    all_coords = curve[["k", "inertia"]].values
    first_point = all_coords[0]
    last_point = all_coords[-1]
    
    line_vec = last_point - first_point
    line_vec_norm = line_vec / np.linalg.norm(line_vec)
    
    vec_from_first = all_coords - first_point
    scalar_proj = np.sum(vec_from_first * line_vec_norm, axis=1)
    
    vec_proj = np.outer(scalar_proj, line_vec_norm)
    vec_to_line = vec_from_first - vec_proj
    dist_to_line = np.linalg.norm(vec_to_line, axis=1)
    
    best_idx = np.argmax(dist_to_line)
    return int(curve["k"].iloc[best_idx])

def silhouette_by_k(X, k_range=config.K_RANGE):
    results = run_kmeans_sweep(X, k_range)
    data = []
    for k, (labels, inertia, model) in results.items():
        if k > 1:
            score = silhouette_score(X, labels, metric=config.DISTANCE_METRIC)
        else:
            score = -1.0
        data.append({"k": k, "silhouette": score})
    return pd.DataFrame(data)

def summarize_k_selection(elbow, silhouette, gap=None):
    df = elbow.merge(silhouette, on="k")
    if gap is not None:
        df = df.merge(gap, on="k")
        
    best_elbow = suggest_k_elbow(elbow)
    best_silhouette = silhouette.loc[silhouette["silhouette"].idxmax(), "k"]
    
    suggestions = {
        "k": [best_elbow, best_silhouette],
        "method": ["Elbow (Max dist)", "Silhouette (Max score)"]
    }
    
    if gap is not None and "gap_value" in gap.columns:
        best_gap = gap.loc[gap["gap_value"].idxmax(), "k"]
        suggestions["k"].append(best_gap)
        suggestions["method"].append("Gap (Max value)")
        
    print("\nSuggested K by method:")
    for m, k in zip(suggestions["method"], suggestions["k"]):
        print(f" - {m}: k={k}")
        
    return df


In [ ]:
# Load datasets
# Make sure these CSV files are available in the same directory if running on Kaggle!
try:
    X_scaled = pd.read_csv("../../data/processed/X_scaled.csv")
    X_unscaled = pd.read_csv("../../data/processed/X_unscaled.csv")
except FileNotFoundError:
    print("Files not found in local directory structure. Assuming Kaggle/Colab current directory.")
    X_scaled = pd.read_csv("X_scaled.csv")
    X_unscaled = pd.read_csv("X_unscaled.csv")

# Drop ID columns before passing to K-Means
cols_to_drop = [c for c in config.ID_COLS if c in X_scaled.columns]
X_train_scaled = X_scaled.drop(columns=cols_to_drop)
X_train_unscaled = X_unscaled.drop(columns=cols_to_drop)

print("Scaled shape:", X_train_scaled.shape)
print("Unscaled shape:", X_train_unscaled.shape)

In [ ]:
# 1. Elbow Method Comparison
df_elbow_scaled = elbow_curve(X_train_scaled)
best_k_elbow_scaled = suggest_k_elbow(df_elbow_scaled)

df_elbow_unscaled = elbow_curve(X_train_unscaled)
best_k_elbow_unscaled = suggest_k_elbow(df_elbow_unscaled)

fig, axes = plt.subplots(1, 2)
axes[0].plot(df_elbow_scaled["k"], df_elbow_scaled["inertia"], marker="o", linestyle="-")
axes[0].axvline(best_k_elbow_scaled, color="r", linestyle="--", label=f"Suggested K = {best_k_elbow_scaled}")
axes[0].set_title("Elbow Curve (SCALED)")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].legend()

axes[1].plot(df_elbow_unscaled["k"], df_elbow_unscaled["inertia"], marker="o", linestyle="-", color="orange")
axes[1].axvline(best_k_elbow_unscaled, color="r", linestyle="--", label=f"Suggested K = {best_k_elbow_unscaled}")
axes[1].set_title("Elbow Curve (UNSCALED)")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Inertia")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 2. Silhouette Score Comparison
df_sil_scaled = silhouette_by_k(X_train_scaled)
best_k_sil_scaled = df_sil_scaled.loc[df_sil_scaled["silhouette"].idxmax(), "k"]

df_sil_unscaled = silhouette_by_k(X_train_unscaled)
best_k_sil_unscaled = df_sil_unscaled.loc[df_sil_unscaled["silhouette"].idxmax(), "k"]

fig, axes = plt.subplots(1, 2)
axes[0].plot(df_sil_scaled["k"], df_sil_scaled["silhouette"], marker="s", color="green", linestyle="-")
axes[0].axvline(best_k_sil_scaled, color="r", linestyle="--", label=f"Suggested K = {best_k_sil_scaled}")
axes[0].set_title("Silhouette Curve (SCALED)")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Average Silhouette Score")
axes[0].legend()

axes[1].plot(df_sil_unscaled["k"], df_sil_unscaled["silhouette"], marker="s", color="purple", linestyle="-")
axes[1].axvline(best_k_sil_unscaled, color="r", linestyle="--", label=f"Suggested K = {best_k_sil_unscaled}")
axes[1].set_title("Silhouette Curve (UNSCALED)")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Average Silhouette Score")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Summary Tables
print("=== SCALED SUMMARY ===")
summary_scaled = summarize_k_selection(df_elbow_scaled, df_sil_scaled)
display(summary_scaled)

print("\n=== UNSCALED SUMMARY ===")
summary_unscaled = summarize_k_selection(df_elbow_unscaled, df_sil_unscaled)
display(summary_unscaled)

In [ ]:
# Based on the curves, choose the best K for both branches
CHOSEN_K_SCALED = max(best_k_elbow_scaled, best_k_sil_scaled)
CHOSEN_K_UNSCALED = max(best_k_elbow_unscaled, best_k_sil_unscaled)

print(f"Applying SCALED K-Means with K = {CHOSEN_K_SCALED}")
print(f"Applying UNSCALED K-Means with K = {CHOSEN_K_UNSCALED}")

labels_scaled, _, model_scaled = run_kmeans(X_train_scaled, k=CHOSEN_K_SCALED)
labels_unscaled, _, model_unscaled = run_kmeans(X_train_unscaled, k=CHOSEN_K_UNSCALED)

# Save labels
pd.DataFrame({"cluster_id": labels_scaled}).to_csv("labels_scaled.csv", index=False)
pd.DataFrame({"cluster_id": labels_unscaled}).to_csv("labels_unscaled.csv", index=False)
print("Saved both labels arrays to local directory.")

In [ ]:
# Describe Centroids physically
# Note: we drop IDs from X_unscaled as well for description
X_unscaled_features = X_unscaled.drop(columns=cols_to_drop)

print("=== SCALED CENTROIDS (in original units) ===")
centroids_physical_scaled = describe_centroids(model_scaled, X_unscaled_features, labels_scaled)
display(centroids_physical_scaled)

print("\n=== UNSCALED CENTROIDS ===")
centroids_physical_unscaled = describe_centroids(model_unscaled, X_unscaled_features, labels_unscaled)
display(centroids_physical_unscaled)

In [ ]:
# Plotting the clusters geographically side-by-side
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Draw pitch bounds and goal line for SCALED
axes[0].plot([0, 120, 120, 0, 0], [0, 0, 80, 80, 0], color="black")
axes[0].plot([120, 120], [36, 44], color="red", linewidth=4)
sns.scatterplot(
    ax=axes[0], x=X_unscaled["location_x"], y=X_unscaled["location_y"], 
    hue=labels_scaled, palette="tab10", alpha=0.6, s=20
)
axes[0].set_title(f"SCALED Shot Zones (K={CHOSEN_K_SCALED})")
axes[0].legend(title="Cluster")

# Draw pitch bounds and goal line for UNSCALED
axes[1].plot([0, 120, 120, 0, 0], [0, 0, 80, 80, 0], color="black")
axes[1].plot([120, 120], [36, 44], color="red", linewidth=4)
sns.scatterplot(
    ax=axes[1], x=X_unscaled["location_x"], y=X_unscaled["location_y"], 
    hue=labels_unscaled, palette="tab10", alpha=0.6, s=20
)
axes[1].set_title(f"UNSCALED Shot Zones (K={CHOSEN_K_UNSCALED})")
axes[1].legend(title="Cluster")

plt.show()